# Lecture 4 — Class Exercise
## Scatter & Bubble Charts: Gapminder

> **Push to:** `week04/lecture04_exercise.ipynb`

**Rules:**
1. Colour used **sparingly** — one categorical variable, no rainbow
2. If showing all continents, either use accessible palette OR grey all + highlight one
3. `size_max` set when using bubble size
4. Log scale for GDP per capita
5. Insight title

---


In [2]:
import pandas as pd
import plotly.express as px


# Dataset: Gapminder — GDP, Life Expectancy, Population by Country
# Source: Gapminder Foundation (gapminder.org)

df = px.data.gapminder()
print(f"Loaded: {len(df)} rows")
print(df.head())

Loaded: 1704 rows
       country continent  year  lifeExp       pop   gdpPercap iso_alpha  \
0  Afghanistan      Asia  1952   28.801   8425333  779.445314       AFG   
1  Afghanistan      Asia  1957   30.332   9240934  820.853030       AFG   
2  Afghanistan      Asia  1962   31.997  10267083  853.100710       AFG   
3  Afghanistan      Asia  1967   34.020  11537966  836.197138       AFG   
4  Afghanistan      Asia  1972   36.088  13079460  739.981106       AFG   

   iso_num  
0        4  
1        4  
2        4  
3        4  
4        4  


In [3]:
# explore

print(df.info())
print("Years:", sorted(df['year'].unique()))
print("Continents:", df['continent'].unique())
print(df.describe().round(1))


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1704 entries, 0 to 1703
Data columns (total 8 columns):
 #   Column     Non-Null Count  Dtype  
---  ------     --------------  -----  
 0   country    1704 non-null   object 
 1   continent  1704 non-null   object 
 2   year       1704 non-null   int64  
 3   lifeExp    1704 non-null   float64
 4   pop        1704 non-null   int64  
 5   gdpPercap  1704 non-null   float64
 6   iso_alpha  1704 non-null   object 
 7   iso_num    1704 non-null   int64  
dtypes: float64(2), int64(3), object(3)
memory usage: 106.6+ KB
None
Years: [np.int64(1952), np.int64(1957), np.int64(1962), np.int64(1967), np.int64(1972), np.int64(1977), np.int64(1982), np.int64(1987), np.int64(1992), np.int64(1997), np.int64(2002), np.int64(2007)]
Continents: ['Asia' 'Europe' 'Africa' 'Americas' 'Oceania']
         year  lifeExp           pop  gdpPercap  iso_num
count  1704.0   1704.0  1.704000e+03     1704.0   1704.0
mean   1979.5     59.5  2.960121e+07     7215.3    

## Task 1 — Scatter: life expectancy change over time

**What to build:** A scatter showing **GDP per capita vs life expectancy** for **two years** (2002 and 2007) to show how both moved — use **colour for year** (just 2 colours), **one continent only**.

Choose any continent except Africa (that was the example). Highlight the change direction.

> 💡 Filter: `df.loc[df['continent'] == 'YOUR_CHOICE']` then filter years


In [9]:
# Define the list of Southeast Asian countries present in the Gapminder dataset
se_asia_countries = [
    'Indonesia', 'Malaysia', 'Philippines', 'Singapore',
    'Thailand', 'Vietnam', 'Myanmar', 'Cambodia'
]

# Filter for Southeast Asian countries and years 2002 and 2007
se_asia_df = df.loc[(df['country'].isin(se_asia_countries)) & (df['year'].isin([2002, 2007]))].copy()

# Convert year to string so it is treated as a discrete/categorical variable
se_asia_df['year'] = se_asia_df['year'].astype(str)

# Create the scatter plot
fig = px.scatter(
    se_asia_df,
    x='gdpPercap',
    y='lifeExp',
    color='year',
    hover_name='country',
    text='country',  # Display country names as text on plot
    log_x=True,
    color_discrete_sequence=['#A6C8E0', '#1F77B4'],  # Two clean, contrasting blues
    labels={
        'gdpPercap': 'GDP per Capita (USD, Log Scale)',
        'lifeExp': 'Life Expectancy (Years)',
        'year': 'Year'
    },
    title='<b>Southeast Asia\'s Growth</b>: Consistent Life Expectancy & Wealth Gains (2002 vs 2007)'
)

# Customize layout
fig.update_layout(
    title_font_size=16,
    plot_bgcolor='white',
    xaxis=dict(showgrid=True, gridcolor='#E5E5E5'),
    yaxis=dict(showgrid=True, gridcolor='#E5E5E5')
)
fig.update_traces(textposition='top center')

fig.show()

## Task 2 — Bubble chart: tell a story

**What to build:** A bubble chart (full 2007 dataset, all countries) where:
- x = GDP per capita (log scale)
- y = life expectancy
- size = population
- colour = ONE continent highlighted (your choice), all others grey
- At least one annotation explaining the highlighted group's story

> This is the grey-and-highlight technique applied to a bubble chart.


In [11]:
# 1. Filter for the year 2007
df_2007 = df[df['year'] == 2007].copy()

# Define the list of Southeast Asian countries
se_asia_countries = [
    'Indonesia', 'Malaysia', 'Philippines', 'Singapore',
    'Thailand', 'Vietnam', 'Myanmar', 'Cambodia'
]

# 2. Define the highlighted group (Southeast Asia) vs Others
df_2007['group'] = df_2007['country'].apply(lambda x: 'Southeast Asia' if x in se_asia_countries else 'Other')

# 3. Sort so that Southeast Asian countries are plotted on top of the grey points
df_2007 = df_2007.sort_values(by='group')

# 4. Create the bubble chart
fig = px.scatter(
    df_2007,
    x='gdpPercap',
    y='lifeExp',
    size='pop',
    color='group',
    hover_name='country',
    log_x=True,
    size_max=50,
    color_discrete_map={
        'Southeast Asia': '#FF7F0E',  # Vibrant orange highlight
        'Other': '#E0E0E0'             # Neutral grey for others
    },
    labels={
        'gdpPercap': 'GDP per Capita (USD, Log Scale)',
        'lifeExp': 'Life Expectancy (Years)',
        'group': 'Region'
    },
    title='<b>Southeast Asia\'s Development Gap</b>: Singapore Outperforms Neighbors in Wealth and Health'
)

# 5. Add custom styling and an annotation pointing out the development gap
fig.update_layout(
    title_font_size=16,
    plot_bgcolor='white',
    xaxis=dict(showgrid=True, gridcolor='#F0F0F0'),
    yaxis=dict(showgrid=True, gridcolor='#F0F0F0'),
    annotations=[
        dict(
            x=4.67,  # Log position of Singapore's high GDP per capita (~47,000 USD)
            y=80,
            xref='x',
            yref='y',
            text='Singapore stands out as an outlier with<br>world-class GDP per capita & longevity<br>compared to the rest of the region.',
            showarrow=True,
            arrowhead=2,
            arrowcolor='#FF7F0E',
            ax=-150,
            ay=40,
            font=dict(color='#FF7F0E', size=11),
            bgcolor='rgba(255, 255, 255, 0.9)',
            bordercolor='#FF7F0E'
        )
    ]
)

fig.show()